# Oncology Wiki — Full Pipeline Test

Test the entire system end-to-end:
1. **Setup** — API key, paths, dependencies
2. **Search** — Gemini grounded search → markdown file
3. **Inspect** — Verify the generated markdown (frontmatter, citations, sources)
4. **Knowledge Graph** — Index the vault, search, traverse, analyze
5. **Explore** — Try your own queries

---
## 1. Setup

In [ ]:
import os
import json
import subprocess
import re
from pathlib import Path

# -- Paths --
PROJECT_ROOT = Path.cwd()
WIKI_DIR = PROJECT_ROOT / "wiki"
RAW_DIR = PROJECT_ROOT / "raw"
SEARCH_DIR = RAW_DIR / "searches"
KG_CLI = PROJECT_ROOT / ".knowledge-graph" / "src" / "cli" / "index.ts"

# -- API Key --
os.environ["GOOGLE_API_KEY"] = "YOUR_KEY_HERE"  # <-- replace this

# -- Knowledge graph env vars --
os.environ["KG_VAULT_PATH"] = str(PROJECT_ROOT)
os.environ["KG_DATA_DIR"] = str(PROJECT_ROOT / ".knowledge-graph-data")

# -- Verify --
assert os.environ["GOOGLE_API_KEY"] != "YOUR_KEY_HERE", "Replace the API key above"
assert KG_CLI.exists(), f"knowledge-graph not found at {KG_CLI} — run: git clone https://github.com/obra/knowledge-graph.git .knowledge-graph && cd .knowledge-graph && npm install"
print("API key is set")
print(f"Project root: {PROJECT_ROOT}")
print(f"KG CLI: {KG_CLI}")

### Helper: run knowledge-graph CLI commands

In [ ]:
def kg(command: str, *args: str, timeout: int = 120) -> dict | list | str:
    """Run a knowledge-graph CLI command and return parsed JSON."""
    cmd = ["npx", "tsx", str(KG_CLI), command, *args]
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=str(PROJECT_ROOT),
        env={**os.environ},
        timeout=timeout,
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        raise RuntimeError(f"kg {command} failed (exit {result.returncode})")
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()

print("kg() helper ready")

---
## 2. Gemini Grounded Search

Run a search query and save the result as a markdown file in `raw/searches/`.

In [ ]:
from search import search

filepath = search("first-line immunotherapy options for metastatic NSCLC with high PD-L1 expression")
print(f"File created: {filepath}")
print(f"File size:    {filepath.stat().st_size:,} bytes")

### Inspect the generated markdown

In [ ]:
import yaml

content = filepath.read_text()
parts = content.split("---", 2)
frontmatter = parts[1].strip() if len(parts) >= 3 else ""
body = parts[2].strip() if len(parts) >= 3 else content

print("=" * 60)
print("FRONTMATTER")
print("=" * 60)
fm = yaml.safe_load(frontmatter)
for k, v in fm.items():
    print(f"  {k}: {repr(v)[:100]}")

print()
print("=" * 60)
print(f"BODY ({len(body)} chars, first 1000 shown)")
print("=" * 60)
print(body[:1000])

### Verify frontmatter and citations

In [ ]:
# Check required fields
required = ["title", "source_type", "search_query", "date_retrieved", "sources"]
for field in required:
    status = "PASS" if field in fm else "FAIL"
    print(f"  [{status}] {field}")

print(f"\n  Source type: {fm.get('source_type')}")
print(f"  Sources count: {len(fm.get('sources', []))}")

# Check inline citations
links = re.findall(r'\[([^\]]+)\]\(https?://[^)]+\)', body)
print(f"  Inline citations: {len(links)}")
for link in links[:5]:
    print(f"    - {link}")
if len(links) > 5:
    print(f"    ... and {len(links) - 5} more")

### Run a second search

In [ ]:
filepath2 = search(
    "KRAS G12C inhibitors clinical trials 2025",
    extra_prompt="focus on sotorasib and adagrasib resistance mechanisms and combination strategies"
)
print(f"File created: {filepath2}")
print(f"File size:    {filepath2.stat().st_size:,} bytes")

### List all search results

In [ ]:
files = sorted(SEARCH_DIR.glob("*.md"))
print(f"Total search files: {len(files)}")
for f in files:
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

---
## 3. Knowledge Graph — Index the Vault

Build the graph from all markdown files and wikilinks in the project. This parses every `.md` file, extracts wikilinks as edges, generates embeddings, and runs Louvain community detection.

**First run downloads a 22MB embedding model — may take a minute.**

In [ ]:
stats = kg("index", timeout=300)
print(json.dumps(stats, indent=2))

---
## 4. Knowledge Graph — Search

### Semantic search (embedding similarity)

In [ ]:
results = kg("search", "immunotherapy NSCLC PD-L1", "--limit", "10")
print(f"Found {len(results)} results:\n")
for r in results:
    score = r.get('score', r.get('similarity', '?'))
    print(f"  [{score:.3f}] {r.get('title', r.get('id', '?'))}")

### Full-text search (keyword matching)

In [ ]:
results_ft = kg("search", "KRAS resistance", "--fulltext", "--limit", "10")
print(f"Found {len(results_ft)} results:\n")
for r in results_ft:
    print(f"  {r.get('title', r.get('id', '?'))}")

---
## 5. Knowledge Graph — Node Inspection

Look up a specific node and see its connections.

In [ ]:
# List all nodes in the graph
results_all = kg("search", "*", "--fulltext", "--limit", "50")
print(f"Total nodes indexed: {len(results_all)}\n")
for r in results_all:
    print(f"  - {r.get('title', r.get('id', '?'))}")

In [ ]:
# Inspect a specific node (use a node name from the list above)
# Change this to any node that exists in your graph
node_name = "index"  # <-- change this to explore other nodes

node = kg("node", node_name, "--full")
print(json.dumps(node, indent=2)[:2000])

---
## 6. Knowledge Graph — Graph Traversal

### Neighbors (N-hop connections)

In [ ]:
# Find everything within 2 hops of a node
node_name = "index"  # <-- change this

neighbors = kg("neighbors", node_name, "--depth", "2")
print(f"Neighbors of '{node_name}' (depth=2):\n")
print(json.dumps(neighbors, indent=2)[:2000])

### Paths between nodes

Find how two entities are connected through the wiki's link structure.

In [ ]:
# Find paths between two nodes
# Change these to nodes that exist in your graph
node_a = "index"   # <-- change this
node_b = "log"     # <-- change this

try:
    paths = kg("paths", node_a, node_b, "--max-depth", "3")
    print(f"Paths from '{node_a}' to '{node_b}':\n")
    print(json.dumps(paths, indent=2)[:2000])
except RuntimeError as e:
    print(f"No paths found or nodes don't exist yet: {e}")

---
## 7. Knowledge Graph — Community Analysis

Louvain community detection groups related pages into clusters.

In [ ]:
communities = kg("communities")
print(f"Detected {len(communities)} communities:\n")
for c in communities:
    print(f"  Community {c['id']}: {c.get('label', '?')} ({c['memberCount']} members)")
    if c.get('summary'):
        print(f"    Summary: {c['summary'][:100]}")

---
## 8. Knowledge Graph — Ranking

### PageRank — most important nodes

In [ ]:
central = kg("central", "--limit", "10")
print("Top 10 nodes by PageRank:\n")
for i, node in enumerate(central, 1):
    print(f"  {i}. {node.get('title', node.get('id', '?'))} (score: {node.get('score', '?')})")

### Bridge nodes — high betweenness centrality

In [ ]:
bridges = kg("bridges", "--limit", "10")
print("Top 10 bridge nodes (connect different parts of the graph):\n")
for i, node in enumerate(bridges, 1):
    print(f"  {i}. {node.get('title', node.get('id', '?'))} (centrality: {node.get('score', '?')})")

---
## 9. Try Your Own Queries

### Search + index in one go

In [ ]:
# 1. Search for something
my_query = "HER2 low breast cancer treatment trastuzumab deruxtecan"  # <-- change this

fp = search(my_query)
print(f"Search saved to: {fp}\n")

# 2. Re-index the graph to pick up the new file
stats = kg("index", timeout=300)
print(f"Index stats: {json.dumps(stats)}\n")

# 3. Search the graph for related content
results = kg("search", my_query, "--limit", "5")
print(f"Graph search results for '{my_query}':")
for r in results:
    score = r.get('score', r.get('similarity', '?'))
    print(f"  [{score:.3f}] {r.get('title', r.get('id', '?'))}")

### Explore a specific node and its neighborhood

In [ ]:
explore_node = "index"  # <-- change to any node name from search results above

print(f"=== Node: {explore_node} ===")
info = kg("node", explore_node)
print(f"Title: {info.get('title')}")
print(f"Outgoing links: {info.get('outgoingCount', 0)}")
print(f"Incoming links: {info.get('incomingCount', 0)}")

if info.get('outgoing'):
    print("\nOutgoing:")
    for e in info['outgoing']:
        print(f"  -> {e.get('target', e.get('targetId', '?'))}")

if info.get('incoming'):
    print("\nIncoming:")
    for e in info['incoming']:
        print(f"  <- {e.get('source', e.get('sourceId', '?'))}")

---
## 10. Cleanup (optional)

Uncomment to delete test search files.

In [ ]:
# Uncomment to delete all search files created during testing:
#
# for f in SEARCH_DIR.glob("*.md"):
#     f.unlink()
#     print(f"Deleted {f.name}")
#
# Uncomment to delete the knowledge graph data:
#
# import shutil
# kg_data = PROJECT_ROOT / ".knowledge-graph-data"
# if kg_data.exists():
#     shutil.rmtree(kg_data)
#     print("Deleted .knowledge-graph-data/")